In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
import shutil
import os
import pandas as pd

In [0]:
params = {
    "input_bcn_id_file":"pharos_donors_for_extraction_20260825.csv",
    "output_excel_file":"pharos_20260828.xlsx",
    "output_parquet_file":"pharos_20260828.parquet"
}

# create text widgets
for k in params.keys():
    dbutils.widgets.text(k, params[k], "")

# fetch values
for k in params.keys():
    params[k] = dbutils.widgets.get(k)
    print(k, ":", params[k])

In [0]:

df_pid = spark.read.csv(os.path.join('/Volumes/7_outgoing/pharos/upload/', params["input_bcn_id_file"]), header=True, inferSchema=True)

bcn_pid = spark.table("6_mgmt.cohorts.bcnb_person_xwalk")
df_pid = df_pid.join(
    bcn_pid,
    df_pid["bcn_id"].cast("string") == bcn_pid["BCNB_No"],
    "left"
).select(["bcn_id", "PERSON_ID"])

display(df_pid.limit(1000))


In [0]:
from pyspark.sql.functions import col

# Load tables
df_examcode = spark.table("4_prod.pacs_dlt.pacs_examcode_dict")
df_imaging = spark.table("4_prod.pacs.imaging_metadata")


# Prepare ec DataFrame
df_ec = df_examcode.filter(
    (
        (col("preferred").ilike("MRI breast%")) & (col("preferred") != "MRI Breast implant Both")
    ) |
    (col("preferred").ilike("XR Mammogram%")) |
    (col("preferred").ilike("US Breast%")) |
    (col("preferred").ilike("US Axilla%"))
).select("short_code").distinct()

# Join all together
df_joined = df_imaging.join(df_pid, df_imaging.PersonId == df_pid.PERSON_ID, "inner") \
    .join(df_ec, df_imaging.ExamCode == df_ec.short_code, "inner")



In [0]:
df_parquet = df_pid.join(df_joined.select(col("AccessionNbr").alias("requested_accession_number"), "PersonId")).drop("PersonID")
output_parquet_path = os.path.join("/Volumes/7_outgoing/pharos/accession_numbers", params["output_parquet_file"])
df_parquet.write.mode("overwrite").parquet(output_parquet_path)

In [0]:
df_acn = df_joined.select(col("AccessionNbr").alias("Accession number")).distinct()

output_acn_path = os.path.join("/Volumes/7_outgoing/pharos/accession_numbers", params["output_excel_file"])


# Write locally on the driver
local_path = os.path.join("/tmp/", params["output_excel_file"])

df_acn_pd = df_acn.toPandas()
df_acn_pd.to_excel(local_path, index=False, engine="openpyxl")

shutil.copy2(local_path, output_acn_path)

display(df_acn.limit(1000))

In [0]:
from pyspark.sql.functions import countDistinct

display(df_joined.groupBy("ExamCode").agg(countDistinct("AccessionNbr").alias("distinct_count_accessionnbr")))